In [ ]:
from langgraph.graph import StateGraph, START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Literal,Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import operator

d:\projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
load_dotenv()


In [ ]:
generator_llm = ChatGroq(model="llama-3.1-8b-instant", 
               temperature=0.3,
                max_tokens=2048,
                api_key=os.getenv("GROQ_API_KEY"))
evaluator_llm = ChatGroq(model="llama-3.1-8b-instant", 
               temperature=0.3,
                max_tokens=2048,
                api_key=os.getenv("GROQ_API_KEY"))
optimizer_llm = ChatGroq(model="llama-3.1-8b-instant", 
               temperature=0.3,
                max_tokens=2048,
                api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved","need_improvement"]= Field(description="The evaluation result of the tweet")
    feedback: str = Field(description="The feedback for improving the tweet, if any")
    # score: int = Field(...,ge=0,le=5,description="The score of the tweet, from 0 to 5")

In [ ]:
structured_evaluation_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [ ]:
# state

class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation: Literal["approved","need_imporvement"]
    iteration:int
    max_iteration:int
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]


In [ ]:
def generate_tweet(state:TweetState):
    # prompt
    message=[
        SystemMessage(content="you are a funny and clever Twitter/X influencer"),
        HumanMessage(content=f"""
Write a short, original and hilarious tweet on the topic : '{state['topic']}'. 
Rules:
-Do not use question answer format.
- max 150 character.
- use observational humor, irony, sarcase, or cultural references.
-think in meme logic, punchlines, or relatable tasks.
- use simple, day to day english
- this is version {state['iteration']+1}.
""")
    ]

    # send generator llm
    response=generator_llm.invoke(message).content

    # return response
    return {'tweet': response,'tweet_history': [response]}

In [ ]:

def evaluate_tweet(state: TweetState):
    message = [
        SystemMessage(
            content="""
You are a ruthless and brutally honest Twitter/X humor critic.

You evaluate tweets based on:
- originality
- humor
- punchiness
- virality potential
- tweet formatting

Your job is to judge whether the tweet genuinely deserves engagement or sounds like low-quality AI-generated content.

You heavily reward:
- sharp punchlines
- meme logic
- relatability
- irony
- observational humor
- strong comedic timing

You heavily penalize:
- generic jokes
- predictable humor
- robotic wording
- motivational tone
- weak punchlines
- cringe formatting
- AI-sounding tweets

If the tweet gets no real laugh, reject it.
"""
        ),

        HumanMessage(
            content=f"""
TOPIC:
"{state['topic']}"

TWEET:
"{state['tweet']}"

THE TWEET MUST FOLLOW THESE RULES:
- Must NOT use question-answer format.
- Must be under 150 characters.
- Must use observational humor, sarcasm, irony, meme logic, cultural references, or relatable everyday situations.
- Must sound natural and human-written.
- Must use simple day-to-day English.
- Must contain a clear punchline, twist, or comedic payoff.
- Must feel tweetable and internet-native.
- Must stay relevant to the topic.

AUTO REJECT IF:
- Uses question-answer/interview/chat format.
- Exceeds 150 characters.
- Sounds robotic, generic, repetitive, motivational, LinkedIn-style, or AI-generated.
- Has no punchline or comedic twist.
- Feels predictable or overused.
- Uses complicated vocabulary or unnatural wording.
- Explains the joke instead of delivering it.
- Is not relevant to the topic.
- Uses cringe hashtags, emoji spam, or forced slang.
- Feels like a caption instead of a tweet.
- Is offensive, hateful, NSFW, or unsafe.

Respond ONLY in this exact format:

evaluation: "approved" or "need_improvement"

feedback: "One concise paragraph explaining the strengths and weaknesses of the tweet. Mention originality, humor quality, punchline strength, virality potential, and formatting issues if any."
"""
        )
    ]

    response =structured_evaluation_llm.invoke(message)

    return {
        'evalation': response.evaluation,'feedback': response.feedback,'feedback_history':[response.feedback]
    }



In [ ]:
def optimize_tweet(state:TweetState):
    message=[
        SystemMessage(content=f"you punch up for the virality and humor based on the given feedbak.")
        HumanMessage(content=f"""
Improve the following tweet based on the feedback. Make it funnier, sharper, and more likely to go viral, while keeping it relevant to the topic.
                     '{state['tweet']}'TweetState
                     Topic: '{state['topic']}'
                     original Tweet: '{state['tweet']}'
                     rewrite it as sohort, viral- worthy tweet, Avoid Q&A stsyle ans stay under 150 character
""")
    ]
    response=optimizer_llm.invoke(message).content
    iteration = state['iteration'] + 1
    return {'tweet': response, 'iteration': iteration,'tweet_history': [response]}


In [ ]:
def route_evaluation(state:TweetState):
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'need_improvement'

In [ ]:
graph=StateGraph(TweetState)

graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)

In [ ]:
graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edge('evaluate',route_evaluation,{'approved': END,'need_improvement':'optimize'})
graph.add_edge('optimize','evaluate')


In [ ]:
workflow = graph.compile()


In [ ]:
workflow

In [ ]:
initial_state = TweetState(topic="Indian Railways ",iteration=1,max_iteration=5)
results=workflow.invoke(initial_state)

In [ ]:
results

In [ ]:
for tweet in results['tweet_history']:
    print(tweet)